In [ ]:
from src.models.components.reverb import ConvolutionalReverb
import torch
from IPython.display import Audio

In [ ]:
reverb = ConvolutionalReverb()
x = torch.randn(1, 1, 192000)
with torch.no_grad():
    y = reverb(x)
with torch.no_grad():
    y_ = reverb.forward_live(x)
y.shape, y_.shape

In [ ]:
Audio(x[0, 0], rate=48000)

In [ ]:
Audio(y[0, 0], rate=48000)

In [ ]:
Audio(y_[0, 0], rate=48000)

In [ ]:
tail = torch.zeros(48000*3)
signals = []
window_size = 512
with torch.no_grad():
    for idx in range(x.shape[-1]//window_size):
        out = reverb.forward_live(x[..., idx*window_size:(idx+1)*window_size])
        signals.append(out[..., :window_size] + tail[..., :window_size]) 
        tail = out[..., window_size:] + torch.cat([tail[..., window_size:], torch.zeros(*tail.shape[:-1], window_size)], dim=-1)

In [ ]:
signal = torch.cat(signals + [tail], dim=-1)

In [ ]:
Audio(signal[0, 0], rate=48000)

In [ ]:
torch.max(torch.abs(signal - y_))